# Data Modeling Task
The goal of this task is to examine the master census metrics JSON file for the CCSVI Dashboard and understand how to transition the existing data into a format usable by an LLM.

### Updates
- master JSON file: `public/data/metrics/census_metrics_by_block_group.json`.
- flattened the nested JSON into a long-form Pandas DataFrame.

# Inspecting the Master JSON

In [107]:
import json
import pandas as pd

In [108]:
# Loading the master .json file
master_file = "../public/data/metrics/census_metrics_by_block_group.json"
with open(master_file, "r") as f:
    data = json.load(f)

In [109]:
# Examining structure
sample_geoid = list(data.keys())[0]
sample = data[sample_geoid]

print("Sample GEOID:", sample_geoid)
print("\nTop-level keys:")
print(sample.keys())

print("\nGeographic Info:")
for key in ["type", "block_group", "census_tract", "county", "state", "population"]:
    print(f"{key}: {sample.get(key)}")

print("\nMetrics datasets:")
print(sample["metrics"].keys())

first_dataset = list(sample["metrics"].keys())[0]
print("\nSample metric names:")
print(list(sample["metrics"][first_dataset].keys())[:10])

Sample GEOID: 5003

Top-level keys:
dict_keys(['type', 'name', 'block_group', 'census_tract', 'county', 'state', 'population', 'metrics'])

Geographic Info:
type: hawaiian_homeland
block_group: None
census_tract: None
county: None
state: None
population: 257

Metrics datasets:
dict_keys(['2022_census_hawaiian_homelands'])

Sample metric names:
['Total Population Under 5', 'Total Population Under 18', 'Total Population Over 65', 'Total population SEX Male', 'Total population SEX Female', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race White', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Black or African American', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race American Indian and Alaska Native', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Asian', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Native Hawaiian and Other Pacific Islander']


In [110]:
# Pivot into long data frame
rows = []

for geoid, area_data in data.items():
    base_info = {
        "geoid": geoid,
        "type": area_data.get("type"),
        "name": area_data.get("name"),
        "block_group": area_data.get("block_group"),
        "census_tract": area_data.get("census_tract"),
        "county": area_data.get("county"),
        "state": area_data.get("state"),
        "population": area_data.get("population")
    }
    
    for dataset_name, metrics in area_data.get("metrics", {}).items():
        for metric_name, values in metrics.items():
            rows.append({
                **base_info,
                "dataset": dataset_name,
                "metric": metric_name,
                "absolute": values.get("absolute"),
                "proportion": values.get("proportion")
            })

df_long = pd.DataFrame(rows)
df_long.head()
print("Shape:", df_long.shape)
type_counts = df_long.groupby("type").size().reset_index(name="row_count")
type_counts

Shape: (44520, 12)


,type,row_count
0,block_group,43320
1,hawaiian_homeland,1200


In [115]:
# Compare metrics between block groups and Hawaiian homelands
metrics_block = set(df_long[df_long["type"] == "block_group"]["metric"].unique())
metrics_hh = set(df_long[df_long["type"] == "hawaiian_homeland"]["metric"].unique())

print("Total Block Group metrics:", len(metrics_block))
print("Total Hawaiian Homeland metrics:", len(metrics_hh))

# Metrics shared by both types
shared_metrics = metrics_block.intersection(metrics_hh)
print("\nShared metrics:", len(shared_metrics))
print(sorted(shared_metrics))

# Metrics unique to Block Groups
block_only = metrics_block - metrics_hh
print("\nMetrics only in Block Groups:", len(block_only))
print(sorted(block_only))

# Metrics unique to Hawaiian Homelands
hh_only = metrics_hh - metrics_block
print("\nMetrics only in Hawaiian Homelands:", len(hh_only))
print(sorted(hh_only))

Total Block Group metrics: 40
Total Hawaiian Homeland metrics: 16

Shared metrics: 0
[]

Metrics only in Block Groups: 40
['American Indian and Alaska Native alone', 'Asian alone', 'Asian and Pacific Island languages: Limited English speaking household', 'Black or African American alone', 'Estimate Aggregate number of vehicles available', 'Estimate Total', 'Female', 'Females Over 65', 'Females Under 18', 'Females Under 5', 'In households: Householder: Female: Living alone', 'In households: Householder: Male: Living alone', 'Institutionalized population', 'Institutionalized population: Correctional facilities for adults', 'Institutionalized population: Juvenile facilities', 'Institutionalized population: Nursing facilities/Skilled-nursing facilities', 'Institutionalized population: Other institutional facilities', 'Male', 'Males Over 65', 'Males Under 18', 'Males Under 5', 'Native Hawaiian and Other Pacific Islander alone', 'No Computer', 'No Health Insurance Coverage', 'No Internet acc

## Structural Observations

1. Two geographic types exist:
   - block_group (standard Census hierarchy — nested within census tracts and counties)
   - hawaiian_homeland (a separate geographic designation that does not follow the tract → block group nesting)

2. Metrics are nested by:
   GEOID → dataset → metric_name → {absolute, proportion}

3. Block groups and Hawaiian Homelands do not have the same metric coverage.
   - Block groups contain 40 metrics.
   - Hawaiian Homelands contain 16 metrics.
   - Some indicators (e.g., institutionalized population, internet access, vehicle access) only exist for block groups.

In [116]:
# Exploring a schema design
df_geo = (
    df_long[df_long["type"] == "block_group"]
    [["geoid", "block_group", "census_tract", "county", "state", "population"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_geo.head()
df_geo = pd.DataFrame(geo_rows)

df_geo.head()

,geoid,block_group,census_tract,county,state,population
0,150010201001,Block Group 1,Census Tract 201,Hawaii County,Hawaii,1826.0
1,150010201002,Block Group 2,Census Tract 201,Hawaii County,Hawaii,1044.0
2,150010201003,Block Group 3,Census Tract 201,Hawaii County,Hawaii,1350.0
3,150010201004,Block Group 4,Census Tract 201,Hawaii County,Hawaii,1213.0
4,150010202021,Block Group 1,Census Tract 202.02,Hawaii County,Hawaii,888.0
